# 05a — Single-Task Sentiment Control: Plain (`noMLM+mean`) & Domain-Adapted (`MLM+attn`)

**Purpose:** Evaluate 4-class human sentiment classification **without intent supervision**, matching the exact hyperparameter budget and frozen splits (`runA`, `runB`, seeds `42, 43, 44`) of the $2 \times 2$ factorial ablation:
1. **`single_task_sent_plain` (`noMLM + mean`):** Unadapted `xlm-roberta-base` + masked mean pooling + 2-layer GELU sentiment head. Completes the symmetric baseline comparison alongside `03_xlmr_plain_baseline`.
2. **`single_task_sent_mlm_attn` (`MLM + attn`):** Reuses cached domain MLM encoder (`mlm_{run}_seed{seed}`) + 4-head single-query attention pooler (`query_sent`) + 2-layer GELU sentiment head. Also exports **both validation and test probability caches** (`preds/stage1_cache_*.npz`) required by the downstream `Sentiment -> Intent` cascade in `05c`.

- Note: If `results/05_single_task_sentiment_{run}_{seed}.json` already exists alongside its `val` probability cache, `05a` reuses it cleanly when `REUSE_05_IF_CACHED = True`; otherwise it trains and populates both `05a` outputs and the Stage-1 cache.

## 1. Environment & Single Config Cell

In [ ]:
# !pip install -q transformers accelerate
# -- Colab / Repo-Root Setup --------------------------------------------------
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(
            ["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"],
            check=True,
        )
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import gc
import math
import time
import json
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

from eval_utils import (
    SENT_CLASSES,
    NS,
    SEEDS,
    log_environment,
    set_all_seeds,
    load_splits,
    macro_f1_present,
    evaluate_and_save,
    summarize_Across_seeds,
)

warnings.filterwarnings("ignore")

# -- Single Config Cell (Identical Hyperparameters to Ablation) ---------------
MODEL_NAME       = "xlm-roberta-base"
MAX_LEN          = 128
MLM_PROB         = 0.15
MLM_EPOCHS       = 3
MLM_BATCH        = 8
MLM_GRAD_ACC     = 2
MLM_LR           = 2e-5

FT_EPOCHS        = 10
FT_BATCH         = 8
FT_GRAD_ACC      = 2
FT_LR            = 2e-5
ENC_LR_MULT      = 0.1
WEIGHT_DECAY     = 0.01
DROPOUT          = 0.3
NUM_ATTN_HEADS   = 4
PATIENCE         = 3

RUNS_TO_EXEC     = ["runA", "runB"]
SEEDS_TO_EXEC    = SEEDS
NOTEBOOK_NAME    = "05a_single_task_sentiment_plain_and_mlm"
REUSE_05_IF_CACHED = False   # Set True only if stage1_cache NPZ files already include val_probs

WORK_DIR         = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV       = None
GOLD_CSV         = None
RESULTS_DIR      = os.path.join(WORK_DIR, "results")
PREDS_DIR        = os.path.join(WORK_DIR, "preds")
ENC_DIR          = os.path.join(WORK_DIR, "mlm_encoders")
for _d in (RESULTS_DIR, PREDS_DIR, ENC_DIR):
    os.makedirs(_d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_info = log_environment()
print("Using device:", DEVICE)

## 2. Dataset & Single-Task Sentiment Models (
oMLM+mean and MLM+attn)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=MLM_PROB)

class SentDS(Dataset):
    def __init__(self, df: pd.DataFrame):
        enc = tok(df["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.ids = enc["input_ids"]
        self.am  = enc["attention_mask"]
        self.s   = df["sent_id"].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return {"input_ids": self.ids[idx], "attention_mask": self.am[idx], "sent": self.s[idx]}

def collate_sent(batch):
    out = tok.pad(
        [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
        return_tensors="pt",
    )
    out["sent"] = torch.tensor([b["sent"] for b in batch], dtype=torch.long)
    return out

class MeanPooler(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        m = mask.unsqueeze(-1).float()
        pooled = (H * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.norm(pooled)

class SingleTaskAttnPooler(nn.Module):
    def __init__(self, hidden: int, heads: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
        self.query = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        B = H.size(0)
        kpm = (mask == 0)
        out, _ = self.attn(self.query.expand(B, -1, -1), H, H, key_padding_mask=kpm)
        return self.norm(out.squeeze(1))

class SingleTaskSentimentModel(nn.Module):
    def __init__(self, enc_path: str, pooling: str = "attn"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool = SingleTaskAttnPooler(h, NUM_ATTN_HEADS, DROPOUT) if pooling == "attn" else MeanPooler(h)
        self.sent_fc = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.sent_head = nn.Linear(h // 2, NS)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        p = self.pool(H, attention_mask)
        return self.sent_head(self.sent_fc(p))

@torch.no_grad()
def predict_sent_probs(model, loader) -> np.ndarray:
    model.eval()
    probs = []
    for b in loader:
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        probs.append(torch.softmax(logits, dim=-1).cpu())
    return torch.cat(probs, dim=0).numpy()

def class_weights(ids, K: int) -> torch.Tensor:
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

## 3. Domain MLM Helper & 3-Seed Training Loop (plain and MLM+attn)

In [ ]:
def get_mlm_encoder(mlm_examples, run: str, seed: int) -> str:
    candidates = [
        os.path.join(ENC_DIR, f"mlm_{run}_seed{seed}"),
        os.path.join(ENC_DIR, f"mlm_seed{seed}"),
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "config.json")):
            print(f"Using cached domain MLM encoder: {c}")
            return c

    out = candidates[0]
    os.makedirs(out, exist_ok=True)
    set_all_seeds(seed)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    m.train()
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(mlm_examples, batch_size=MLM_BATCH, shuffle=True, collate_fn=mlm_collator, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=MLM_LR, weight_decay=WEIGHT_DECAY)
    total = MLM_EPOCHS * math.ceil(len(loader) / MLM_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

    for ep in range(MLM_EPOCHS):
        tl, n = 0.0, 0
        opt.zero_grad()
        for step, b in enumerate(loader):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            if (b["labels"] != -100).any():
                loss = m(**b).loss
                (loss / MLM_GRAD_ACC).backward()
                tl += loss.item()
                n += 1
            if (step + 1) % MLM_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad()
        print(f"  [MLM {run} seed {seed}] epoch {ep+1}/{MLM_EPOCHS} loss {tl/max(n,1):.4f}")

    m.base_model.save_pretrained(out)
    del m, opt, sched
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

VARIANTS = [
    ("single_task_sent_plain",    "05a_single_task_sentiment_plain",    False, "mean"),
    ("single_task_sent_mlm_attn", "05a_single_task_sentiment_mlm_attn", True,  "attn"),
]

summaries = []

for run in RUNS_TO_EXEC:
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    mlm_texts = train_df["text"].tolist() + val_df["text"].tolist()
    assert not (set(mlm_texts) & set(test_df["text"])), "Leakage guard: test text in MLM corpus!"
    _enc = tok(mlm_texts, truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)
    mlm_examples = [
        {"input_ids": a, "attention_mask": b, "special_tokens_mask": c}
        for a, b, c in zip(_enc["input_ids"], _enc["attention_mask"], _enc["special_tokens_mask"])
    ]

    train_ds, val_ds, test_ds = SentDS(train_df), SentDS(val_df), SentDS(test_df)
    val_loader  = DataLoader(val_ds,  batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_sent)
    test_loader = DataLoader(test_ds, batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_sent)
    W_SENT = class_weights(train_df["sent_id"].values, NS).to(DEVICE)
    val_ys = val_df["sent_id"].values

    for model_tag, nb_tag, use_mlm, pooling in VARIANTS:
        seed_payloads = []
        for seed in SEEDS_TO_EXEC:
            cache_npz = os.path.join(PREDS_DIR, f"stage1_cache_{model_tag}_{run}_{seed}.npz")
            existing_json = os.path.join(RESULTS_DIR, f"05_single_task_sentiment_{run}_{seed}.json")

            if (
                REUSE_05_IF_CACHED
                and model_tag == "single_task_sent_mlm_attn"
                and os.path.exists(existing_json)
                and os.path.exists(cache_npz)
            ):
                with open(existing_json, "r", encoding="utf-8") as f:
                    p05 = json.load(f)
                p05_copy = dict(p05)
                p05_copy["notebook"] = nb_tag
                p05_copy["model"] = model_tag
                with open(os.path.join(RESULTS_DIR, f"{nb_tag}_{run}_{seed}.json"), "w", encoding="utf-8") as f:
                    json.dump(p05_copy, f, indent=2)
                seed_payloads.append(p05_copy)
                print(f"[Reuse] {model_tag} ({run}, seed={seed}) from 05_single_task_sentiment")
                continue

            enc_path = get_mlm_encoder(mlm_examples, run=run, seed=seed) if use_mlm else MODEL_NAME
            set_all_seeds(seed)
            model = SingleTaskSentimentModel(enc_path, pooling=pooling).to(DEVICE)
            g = torch.Generator()
            g.manual_seed(seed)
            loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate_sent, generator=g)

            crit_s = nn.CrossEntropyLoss(weight=W_SENT)
            head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
            opt = torch.optim.AdamW(
                [{"params": model.encoder.parameters(), "lr": FT_LR * ENC_LR_MULT},
                 {"params": head_params, "lr": FT_LR}],
                weight_decay=WEIGHT_DECAY,
            )
            total = FT_EPOCHS * math.ceil(len(loader) / FT_GRAD_ACC)
            sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

            best_val, best_ep, best_state, bad = -1.0, 0, None, 0
            t0 = time.time()
            for ep in range(FT_EPOCHS):
                model.train()
                tl = 0.0
                opt.zero_grad()
                for step, b in enumerate(loader):
                    logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
                    loss = crit_s(logits, b["sent"].to(DEVICE))
                    (loss / FT_GRAD_ACC).backward()
                    tl += loss.item()
                    if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(loader):
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        opt.step()
                        sched.step()
                        opt.zero_grad()
                vs = predict_sent_probs(model, val_loader)
                f_s = macro_f1_present(val_ys, vs.argmax(1), NS)
                print(f"  [{model_tag} {run} s{seed}] ep {ep+1:>2} | train {tl/len(loader):.4f} | val Sent F1 {f_s:.3f}")
                if f_s > best_val:
                    best_val, best_ep, bad = f_s, ep + 1, 0
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    bad += 1
                    if bad >= PATIENCE:
                        print("  early stop")
                        break

            elapsed_s = round(time.time() - t0, 2)
            model.load_state_dict(best_state)
            vs_best = predict_sent_probs(model, val_loader)
            ts_best = predict_sent_probs(model, test_loader)

            # Export Stage-1 val + test caches for 05c cascading/pipeline reuse
            np.savez_compressed(
                cache_npz,
                val_probs=vs_best.astype(np.float32),
                test_probs=ts_best.astype(np.float32),
                val_pred=vs_best.argmax(1).astype(np.int64),
                test_pred=ts_best.argmax(1).astype(np.int64),
            )

            del model, opt, sched, best_state
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            payload = evaluate_and_save(
                notebook_name=nb_tag,
                model_name=model_tag,
                run=run,
                seed=seed,
                test_df=test_df,
                pred_sent_id=ts_best.argmax(1),
                pred_intent_id=None,
                sent_probs=ts_best,
                intent_probs=None,
                extra_meta={
                    "use_mlm": use_mlm,
                    "pooling": pooling,
                    "best_epoch": best_ep,
                    "best_val_sent_f1": round(float(best_val), 4),
                    "train_seconds": elapsed_s,
                    "stage1_cache": cache_npz,
                },
                results_dir=RESULTS_DIR,
                preds_dir=PREDS_DIR,
            )
            seed_payloads.append(payload)

            if seed == 42:
                print(f"\n--- Per-Class Gold Sentiment Report ({model_tag}, {run}, seed=42) ---")
                print(classification_report(test_df["gold_sent_id"].values, ts_best.argmax(1), target_names=SENT_CLASSES, digits=3, zero_division=0))

        summaries.append(summarize_Across_seeds(seed_payloads))

df_sum = pd.DataFrame(summaries)
display(df_sum[["run", "model", "n_seeds", "sent_acc_str", "sent_f1_3cls_str", "sent_f1_4cls_str"]])
with open(os.path.join(RESULTS_DIR, f"{NOTEBOOK_NAME}_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summaries, f, indent=2)

## 4. Diagnostic Comparison vs. Joint MLM+attn & Package Outputs

In [ ]:
JOINT_REF = {
    "runA": {"sent_acc": 0.776, "sent_f1_3cls": 0.719, "sent_f1_4cls": 0.539},
    "runB": {"sent_acc": 0.751, "sent_f1_3cls": 0.683, "sent_f1_4cls": 0.512},
}
for row in summaries:
    ref = JOINT_REF.get(row["run"], {})
    if ref and row.get("sent_acc_mean") is not None:
        print(
            f"[{row['run']} | {row['model']}] "
            f"Sent Acc={row['sent_acc_str']} (vs Joint {ref['sent_acc']:.3f}: {row['sent_acc_mean'] - ref['sent_acc']:+.3f}) | "
            f"3-cls F1={row['sent_f1_3cls_str']} (vs Joint {ref['sent_f1_3cls']:.3f}: {row['sent_f1_3cls_mean'] - ref['sent_f1_3cls']:+.3f})"
        )

zip_path = os.path.join(WORK_DIR, f"{NOTEBOOK_NAME}_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (RESULTS_DIR, PREDS_DIR):
        for fp in sorted(glob.glob(os.path.join(folder, "*05a*")) + glob.glob(os.path.join(folder, "*single_task_sent*")) + glob.glob(os.path.join(folder, "stage1_cache_*"))):
            zf.write(fp, arcname=os.path.relpath(fp, WORK_DIR))
print(f"Packaged outputs into: {zip_path}")
# Optional Google Drive copy in Colab:
# import shutil; shutil.copy(zip_path, "/content/drive/MyDrive/")